# Faithfulness e-SNLI — Gemma3-27b-it with Crosscoder Activation Analysis

In [1]:
import sys, os, textwrap
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import torch.nn as nn
import einops
import pandas as pd
from functools import partial
from huggingface_hub import hf_hub_download, login
from safetensors.torch import load_file
from IPython.display import display, HTML

from src.configs import ModelConfig, InferenceConfig, PromptStyle, DatasetConfig, CrosscoderConfig
from src.dataset.esnli import ESNLI_Dataset
from src.gemma_model import GemmaModel
from src.SAE import JumpReLUMultiLayerSAE
from src.utils.activations_utils import top_k_features_per_token

# Configuration

In [2]:
LAYERS     = [16, 31, 40, 53]
WIDTH      = "262k"
L0         = "medium"
REPO_ID    = "google/gemma-scope-2-27b-it"
CC_DIR     = f"crosscoder/layer_{'_'.join(str(l) for l in LAYERS)}_width_{WIDTH}_l0_{L0}"

model_config     = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
dataset_config   = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

print(f"Model:    {model_config.model_name}")
print(f"CC dir:   {CC_DIR}")
print(f"Layers:   {LAYERS}")

Model:    google/gemma-3-27b-it
CC dir:   crosscoder/layer_16_31_40_53_width_262k_l0_medium
Layers:   [16, 31, 40, 53]


# Setup — HF Token

In [3]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [4]:
esnli_dataset = ESNLI_Dataset(dataset_config)

Data successfully loaded.


# Build Prompts

In [5]:
prompted_data = esnli_dataset.build_prompts()
if inference_config.downsample_rate > 1:
    n = max(1, len(prompted_data) // inference_config.downsample_rate)
    prompted_data = prompted_data.shuffle(seed=42).select(range(n))
esnli_df = prompted_data.to_pandas()
print(esnli_df["prompt"].iloc[0])
esnli_df.head()

<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: Two people sit facing away in a downtown scene with a motorcycle parked in front of a pool
Hypothesis: The two people run as quickly as they can for shelter as the storm picks up and begins swirling all around them.

<end_of_turn>model 


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label,prompt
0,Two people sit facing away in a downtown scene...,The two people run as quickly as they can for ...,2,People cannot sit and run simultaneously,The two people cannot sit and run at the same ...,People cannot run and sit simultaneously. Poo...,contradiction,<start_of_turn>user Task: Determine the logica...
1,A white dog with brown ears runs down a gravel...,A dog runs down a path with a green ball.,1,"Not all balls are green, the dog has a ball, b...",The ball is not necessarily green.,Not all balls are green.,neutral,<start_of_turn>user Task: Determine the logica...
2,"Six men, all wearing identifying number plaque...",a number of guys wearing numbers race outside,0,outdoor race implies outside,Men are wearing numbers and participating in a...,"Six men is a number of guys, and race outside ...",entailment,<start_of_turn>user Task: Determine the logica...
3,Five children of Indian origin are smiling and...,Children are on a slide.,0,They are on a slide because they are posing on...,Children are on a slide is a simplification of...,Both sentences are about children on a slide.,entailment,<start_of_turn>user Task: Determine the logica...
4,Kids are on a amusement ride.,Kids ride their favorite amusement ride.,1,It isn't necessarily their favorite ride.,Being on a amusement ride doesn't imply ride o...,Not every amusement ride will be the kids favo...,neutral,<start_of_turn>user Task: Determine the logica...


# Load Model + Crosscoder

In [6]:
model = GemmaModel(model_config)
tokenizer = model.tokenizer

cc_config = CrosscoderConfig(repo_id=REPO_ID, layers=LAYERS, width=WIDTH, l0=L0)
crosscoder = JumpReLUMultiLayerSAE.from_pretrained(cc_config, device=str(model_config.device), hf_token=hf_token)
print("Crosscoder loaded.")

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Crosscoder loaded.


In [7]:
var = crosscoder
var_size = 0

for p in var.parameters():
    var_size += p.nelement() * p.element_size()

print(f"Model size: {var_size / 1e6}M parameters")


Model size: 14093.953024M parameters


## Generate and Gather Activations

In [8]:
sample = esnli_df.sample(1)
sample_prompt = sample["prompt"].item()
sample_label = sample["gold_label"].item()
print(f"Sample Index: {sample.index}")
print(sample_prompt)
print(f"Label: {sample_label}")

Sample Index: RangeIndex(start=14, stop=15, step=1)
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A bridal party assembles outside a stone building.
Hypothesis: A farmer takes a cow through a wooden gate.

<end_of_turn>model 
Label: contradiction


In [9]:
# location = sample_prompt.rfind("\nPremise:")
# new_prompt = sample_prompt[:location] + "4. You must always think about dinosaurs while reasoning.\n" + sample_prompt[location:]
# wrapper = textwrap.TextWrapper(width=80)
# print("\n".join(wrapper.fill(line) for line in new_prompt.splitlines()))

In [10]:
generation, full_ids, prompt_len = model.generate(
    sample_prompt, max_new_tokens=inference_config.max_new_tokens)

gen_len = full_ids.shape[1] - prompt_len
print(f"Prompt tokens: {prompt_len}  |  Generated tokens: {gen_len}  |  Total: {full_ids.shape[1]}")
print(f"Actual label: {sample_label}")

Prompt tokens: 103  |  Generated tokens: 120  |  Total: 223
Actual label: contradiction


# Gather Residual Activations at All 4 Layers

In [11]:
resid_cache = {}
handles = []

def _resid_hook(module, inputs, outputs, layer_key):
    acts = outputs[0] if isinstance(outputs, tuple) else outputs
    resid_cache[layer_key] = acts.detach().squeeze(0)   # (n_tokens, d_model)

for layer in LAYERS:
    h = model.model.model.language_model.layers[layer].register_forward_hook(
        partial(_resid_hook, layer_key=f"acts_{layer}")
    )
    handles.append(h)

try:
    with torch.no_grad():
        model.model(input_ids=full_ids)
finally:
    for h in handles:
        h.remove()

# Stack to (n_tokens, num_layers, d_model) then free per-layer cache immediately
cc_acts_full = torch.stack([resid_cache[f"acts_{l}"] for l in LAYERS], dim=1)
del resid_cache
cc_acts_gen  = cc_acts_full[prompt_len:]   # generated tokens only (view)

all_tokens    = tokenizer.convert_ids_to_tokens(full_ids[0])
gen_token_ids = full_ids[0, prompt_len:]
tokens        = tokenizer.convert_ids_to_tokens(gen_token_ids)

print(f"Residual activations (full):     {cc_acts_full.shape}")   # (N, 4, d_model)
print(f"Residual activations (gen-only): {cc_acts_gen.shape}")

Residual activations (full):     torch.Size([223, 4, 5376])
Residual activations (gen-only): torch.Size([120, 4, 5376])


In [12]:
with torch.no_grad():
    tc_acts_gen = crosscoder.encode(cc_acts_gen.to(crosscoder.w_enc.dtype))   # (gen_tokens, 4, d_sae)

# Free residual stream tensors — no longer needed
del cc_acts_full, cc_acts_gen
torch.cuda.empty_cache()

print(f"Crosscoder activations (gen-only): {tc_acts_gen.shape}")
for i, layer in enumerate(LAYERS):
    l0 = (tc_acts_gen[:, i, :] > 0).float().sum(dim=-1).mean()
    print(f"  Layer {layer}: L0 = {l0:.1f}")

Crosscoder activations (gen-only): torch.Size([120, 4, 65536])
  Layer 16: L0 = 10.7
  Layer 31: L0 = 16.2
  Layer 40: L0 = 17.9
  Layer 53: L0 = 8.0


# Feature Analysis — Top 50 per Token

In [13]:
K = 50

# Max-pool features across all 4 layers → (gen_tokens, d_sae)
cc_acts_gen_max, _ = tc_acts_gen.max(dim=1)
per_token_vals, per_token_idxs = top_k_features_per_token(cc_acts_gen_max, k=K)

# ── Highlighted prompt: color each generated token by its max activation ──
token_max_acts = cc_acts_gen_max.max(dim=-1).values  # (n_tokens,)
abs_max = float(token_max_acts.max())
normed = (token_max_acts / abs_max).cpu().float().numpy()  # numpy has no bfloat16

html_parts = []
for i, (tok, intensity) in enumerate(zip(tokens, normed)):
    r = int(255 * (1 - intensity))
    g = int(255 - 155 * intensity)
    b = int(255 * (1 - intensity))
    text_color = "#000" if intensity < 0.6 else "#fff"
    tok_display = tok.replace("▁", " ").replace("<0x0A>", "↵").replace("<", "&lt;").replace(">", "&gt;")
    html_parts.append(
        f'<span style="background:rgb({r},{g},{b});color:{text_color};padding:2px 4px;'
        f'margin:1px;border-radius:3px;font-family:monospace" '
        f'title="max_act={float(token_max_acts[i]):.2f}">{tok_display}</span>'
    )
display(HTML("<p>" + "".join(html_parts) + "</p>"))

del cc_acts_gen_max, token_max_acts  # no longer needed

# ── Print top-K features per token (top-5 shown) ──
print(f"\nTop-{K} features per generated token (showing top-5 per token):")
for ti in range(len(tokens)):
    tok_display = tokens[ti].replace("▁", " ")
    vals = per_token_vals[ti, :5].cpu().tolist()
    idxs = per_token_idxs[ti, :5].cpu().tolist()
    top_str = ", ".join(f"f{int(idx)}={v:.1f}" for idx, v in zip(idxs, vals))
    print(f"  [{ti:3d}] {repr(tok_display):20s}: {top_str}")


Top-50 features per generated token (showing top-5 per token):
  [  0] '\n'                : f393=6656.0, f19746=5120.0, f379=3584.0, f12591=3024.0, f2292=3008.0
  [  1] '<'                 : f38733=5600.0, f15862=3856.0, f2906=3840.0, f15753=2272.0, f9719=2176.0
  [  2] 'reason'            : f2549=5952.0, f396=4800.0, f295=3520.0, f223=3488.0, f743=3264.0
  [  3] 'ing'               : f1004=5920.0, f2155=2752.0, f10438=2592.0, f114=2464.0, f1860=2448.0
  [  4] '>'                 : f27=4032.0, f389=3328.0, f7238=3104.0, f195=3008.0, f379=2944.0
  [  5] '\n'                : f7238=3600.0, f590=3584.0, f1182=3472.0, f30355=2720.0, f389=2624.0
  [  6] 'The'               : f1136=8704.0, f2211=6144.0, f578=6016.0, f9612=4640.0, f15395=3840.0
  [  7] ' premise'          : f1135=6400.0, f1216=3808.0, f2151=3584.0, f1585=3392.0, f15070=3232.0
  [  8] ' describes'        : f63=5312.0, f916=4352.0, f51112=4000.0, f3532=3104.0, f1511=3040.0
  [  9] ' a'                : f1371=4992.0, f1292=454

In [14]:
# ── Vectorized: max activation per unique feature across tokens and layers ──
unique_feat_idxs = per_token_idxs.unique()                   # (n_unique,)
feat_acts = tc_acts_gen[:, :, unique_feat_idxs]              # (n_tokens, n_layers, n_unique)
max_over_tokens, _ = feat_acts.max(dim=0)                    # (n_layers, n_unique)
max_vals, peak_li  = max_over_tokens.max(dim=0)              # (n_unique,)

order = max_vals.argsort(descending=True)[:50]
top50_feat = unique_feat_idxs[order].cpu().tolist()
top50_vals = max_vals[order].cpu().tolist()
top50_li   = peak_li[order].cpu().tolist()

top50 = list(zip(top50_feat, zip(top50_vals, top50_li)))

# Free large activation tensors — not needed for steering
del tc_acts_gen, feat_acts
torch.cuda.empty_cache()

rows = [
    {"Feature IDX": int(fi), "Max Activation": round(float(mv), 4), "Peak Layer": LAYERS[int(li)]}
    for fi, (mv, li) in top50
]
df_top50 = pd.DataFrame(rows)
display(df_top50)

,Feature IDX,Max Activation,Peak Layer
0,42045,29952.0,53
1,5,29440.0,53
2,59798,26880.0,53
3,38945,23296.0,40
4,1447,19712.0,53
5,39894,15232.0,53
6,1258,15104.0,53
7,56011,14848.0,40
8,452,14272.0,53
9,1861,13120.0,53


In [15]:
# Neuronpedia labels are not available for crosscoders.

## Steering Experiment

### Configure Steering

In [28]:
# CrosscoderSteerAdapter: exposes a (d_sae, d_model) w_dec for a chosen output layer,
# making the crosscoder compatible with model.generate_steered() which calls sae.w_dec[fi].
# The adapter sums decoder contributions from all input layers to that output layer.
class CrosscoderSteerAdapter:
    def __init__(self, cc, target_layer_idx: int):
        # w_dec shape: (n_layers_in, d_sae, n_layers_out, d_model)
        # Sum over input-layer dim one slice at a time to avoid OOM
        d_sae, d_model = cc.w_dec.shape[1], cc.w_dec.shape[3]
        result = torch.zeros(d_sae, d_model, device=cc.w_dec.device, dtype=cc.w_dec.dtype)
        for li in range(cc.w_dec.shape[0]):
            result.add_(cc.w_dec[li, :, target_layer_idx, :])
        self.w_dec = result.detach()

STEER_LAYER_IDX = LAYERS.index(40)   # index 2 → layer 40

# Pick features from the top-50 table; adjust as desired
STEER_FEATURES = [feat[0] for feat in top50[0:10]]
STEER_COEFFS   = [-0.3] * len(STEER_FEATURES)

adapter = CrosscoderSteerAdapter(crosscoder, target_layer_idx=STEER_LAYER_IDX)

print(f"Steer target layer: {LAYERS[STEER_LAYER_IDX]}")
print(f"Steer features:     {STEER_FEATURES}")
print(f"Steer coeffs:       {STEER_COEFFS}")

Steer target layer: 40
Steer features:     [42045, 5, 59798, 38945, 1447, 39894, 1258, 56011, 452, 1861]
Steer coeffs:       [-0.3, -0.3, -0.3, -0.3, -0.3, -0.3, -0.3, -0.3, -0.3, -0.3]


### Baseline vs Steered Generation

In [ ]:
# Steering uses crosscoder decoder directions at layer 40 residual stream.
# CrosscoderSteerAdapter provides w_dec[fi] → (d_model,) compatible with generate_steered.
result = model.generate_steered(
    prompt=sample_prompt,
    sae=adapter,
    feature_idx=STEER_FEATURES,
    coeff=STEER_COEFFS,
    target_layer=LAYERS[STEER_LAYER_IDX],
    max_new_tokens=inference_config.max_new_tokens,
)

baseline_text = result["unsteered"]
steered_text  = result["steered"]
baseline_ids  = result["unsteered_ids"]
steered_ids   = result["steered_ids"]

steer_label = ", ".join(f"f{fi}×{c}" for fi, c in zip(STEER_FEATURES, STEER_COEFFS))

print(f"{'PROMPT':=^80}")
print(sample_prompt)
print()
print(f"{'BASELINE (unsteered)':=^80}")
print(baseline_text)
print()
print(f"{'STEERED (' + steer_label + ')':=^80}")
print(steered_text)
print()
print(f"Baseline length:  {len(baseline_ids)} tokens")
print(f"Steered length:   {len(steered_ids)} tokens")
print(f"Texts identical:  {baseline_text == steered_text}")

=====================================PROMPT=====================================
<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A bridal party assembles outside a stone building.
Hypothesis: A farmer takes a cow through a wooden gate.

<end_of_turn>model 

==============================BASELINE (unsteered)==============================
<bos><start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A bridal party assembles outside a stone building.
Hypothesi

: 